# Submission Notebook

Generates submission.csv from blended predictions.

CPU only. Run after 03_inference.ipynb.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

# Config
DATA_DIR    = Path('../data/raw')
RESULT_DIR  = Path('../results/experiments')
SUB_DIR     = Path('../results/submissions')
SUB_DIR.mkdir(parents=True, exist_ok=True)

# Point to the blended predictions file from 03_inference.ipynb
PRED_FILE   = RESULT_DIR / 'blend_EXP-001_test.npy'   # update as needed
TARGET_COL  = 'target'   # column name in submission
ID_COL      = 'id'       # ID column in test.csv

print(f'PRED_FILE = {PRED_FILE}')

In [ ]:
# Load test IDs
test_path = DATA_DIR / 'test.csv'
sample_sub_path = DATA_DIR / 'sample_submission.csv'

if sample_sub_path.exists():
    sub = pd.read_csv(sample_sub_path)
    print(f'Loaded sample_submission.csv: {sub.shape}')
elif test_path.exists():
    test = pd.read_csv(test_path)
    sub = pd.DataFrame({ID_COL: test[ID_COL], TARGET_COL: 0})
    print(f'Built submission from test.csv: {sub.shape}')
else:
    # Demo fallback
    print('WARNING: No data files. Creating demo submission.')
    sub = pd.DataFrame({ID_COL: range(100), TARGET_COL: 0.5})

print(sub.head(3))

In [ ]:
# Load predictions
if PRED_FILE.exists():
    preds = np.load(PRED_FILE)
    print(f'Predictions shape: {preds.shape}')
    print(f'Predictions range: [{preds.min():.4f}, {preds.max():.4f}]')
    assert len(preds) == len(sub), \
        f'Mismatch: preds={len(preds)}, submission={len(sub)}'
    sub[TARGET_COL] = preds
else:
    print(f'PRED_FILE not found: {PRED_FILE}')
    print('Run 02_training.ipynb + 03_inference.ipynb first.')
    raise FileNotFoundError(PRED_FILE)

In [ ]:
# Validate and save
print('=== Submission Preview ===')
print(sub.head())
print(f'\nShape: {sub.shape}')
print(f'Missing values: {sub.isnull().sum().sum()}')
print(f'Target stats: mean={sub[TARGET_COL].mean():.4f}  std={sub[TARGET_COL].std():.4f}')

# Save with timestamp
ts       = datetime.now().strftime('%Y%m%d_%H%M')
out_path = SUB_DIR / f'submission_{ts}.csv'
sub.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')

# Also save as latest
latest_path = SUB_DIR / 'submission_latest.csv'
sub.to_csv(latest_path, index=False)
print(f'Saved: {latest_path}')

print('\nNext steps:')
print('1. Upload to Kaggle')
print('2. Record Public LB score in memory/leaderboard_progress.md')
print('3. Run reviewer_prompt.md to analyze results')